In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 09b · Inferencia — del modelo registrado a una app que cualquiera puede usar

> Notebook del proyecto (versión «b», datos de este caso). Plantilla del curso: `09_inferencia.ipynb` (sin modificar). Ejemplo: `material-del-curso/ejemplo-creditsight/notebooks/09_inferencia.ipynb`.

El modelo registrado vive en MLflow; **la app no consulta MLflow ni Databricks**. Esta etapa construye el puente entre el registro y una persona real, y deja la app lista para correr **dentro de Databricks (Databricks Apps)**:

1. **Exportar** el modelo `champion` a un archivo (`models/model.joblib`).
2. Escribir el **esquema de entrada** (`models/input_schema.json`): qué columnas, qué valores son *posibles*, qué categorías, perfiles de ejemplo.
3. Probar la función de predicción con datos buenos, incompletos e **imposibles**.
4. Comprobar que el archivo exportado **es** el modelo que se evaluó en la etapa 7.
5. Verificar la **ficha del modelo**.
6. **Ejecutar la app de Streamlit** de forma automática (`AppTest`) con los perfiles de ejemplo.
7. **Armar el paquete de Databricks Apps** y comprobar que corre solo, sin el resto del repositorio.

## 0. Datos y rutas

In [2]:
import json

import joblib
import numpy as np
import pandas as pd

from proyecto.data.cleaning import TARGET, VALID_RANGES
from proyecto.features.engineering import STAGE4_DROP, feature_columns
from proyecto.models.train import FALSE_NEGATIVE_COST, FALSE_POSITIVE_COST, decision_threshold

ROOT = Path.cwd().resolve().parent
REPORTS, MODELS, PROCESSED = ROOT / "reports", ROOT / "models", ROOT / "data" / "processed"

train = joblib.load(PROCESSED / "train.joblib")            # solo entrenamiento: la prueba no se usa para decidir nada aquí
NUMERICAS, CATEGORICAS = feature_columns(drop=STAGE4_DROP)
COLUMNAS = NUMERICAS + CATEGORICAS
assert len(COLUMNAS) == 28 and TARGET not in COLUMNAS

# El modelo guarda sus clases como 0 (Pass) y 1 (Fail): predict_proba devuelve [P(Pass), P(Fail)].
CLASES = ["Pass", "Fail"]
# Filas = clase real, columnas = clase decidida: no avisar a quien pierde cuesta 5; avisar a quien aprueba, 1 (supuesto).
MATRIZ_DE_COSTOS = [[0, FALSE_POSITIVE_COST], [FALSE_NEGATIVE_COST, 0]]
UMBRAL = decision_threshold()                               # 1 / (1 + 5) = 1/6

# Lo que es POSIBLE en el mundo real (conocimiento del analista, no de los datos): fuera de estos rangos la app se niega.
RANGOS_POSIBLES = {
    **{k: v for k, v in VALID_RANGES.items() if k in COLUMNAS},   # los de data/README.md (limpieza, etapa 3)
    "practice_tests_completed": (0, 200),
    "stress_level": (1, 10),
    "exam_preparation_days": (0, 365),
    "time_management_score": (0, 100),
    "exam_anxiety_level": (1, 10),
}
decisiones = json.loads((REPORTS / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
resultados = json.loads((REPORTS / "07_metricas_test.json").read_text(encoding="utf-8"))
registro = json.loads((REPORTS / "08_registro.json").read_text(encoding="utf-8"))
assert abs(decisiones["umbral"] - UMBRAL) < 1e-12, "el umbral de la app no es el congelado en la etapa 7"
print("entrada:", len(NUMERICAS), "numéricas y", len(CATEGORICAS), "categóricas · clases", CLASES, "· umbral", round(UMBRAL, 4))
print("modelo registrado:", registro["modelo_registrado"], "v" + str(registro["version"]), "alias", registro["alias"])

entrada: 12 numéricas y 16 categóricas · clases ['Pass', 'Fail'] · umbral 0.1667
modelo registrado: estudiantes_aprueba_o_no v1 alias champion


## 1. Exportar el modelo `champion`

La fuente de verdad es el **registro**: se pide el modelo **por su alias** y se guarda en un archivo, para tener la certeza de que se entrega *lo que quedó registrado* y no otra copia. Si el registro no responde, se usa la copia local de la etapa 7 y el notebook lo dice, para que nadie crea que usa una cosa y use otra.

In [3]:
modelo, origen = None, None
try:
    import mlflow
    from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name

    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        donde = configure_mlflow()
    NOMBRE = registered_model_name(registro["modelo_registrado"])
    modelo = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")
    origen = f"registro de MLflow ({donde}), {NOMBRE}@champion"
except Exception as error:
    print("No se pudo leer el registro:", str(error).splitlines()[0][:110])

if modelo is None:
    modelo = joblib.load(MODELS / "final_pipeline.joblib")
    origen = "copia local de la etapa 7 (el registro no respondió)"

joblib.dump(modelo, MODELS / "model.joblib", compress=3)
print("models/model.joblib guardado desde:", origen)
print(f"tamaño: {(MODELS / 'model.joblib').stat().st_size / 1e3:.1f} KB (muy por debajo del límite de 100 MB de GitHub)")

models/model.joblib guardado desde: registro de MLflow (local), estudiantes_aprueba_o_no@champion
tamaño: 3.7 KB (muy por debajo del límite de 100 MB de GitHub)


## 2. El esquema de entrada

La app y la función de predicción necesitan saber qué se puede preguntar. Dos cosas distintas:

- **Rango observado:** lo visto en el entrenamiento. Fuera de él solo se **avisa**.
- **Rango posible** (`RANGOS_POSIBLES`): lo que puede ocurrir en el mundo real. Fuera de él la predicción **se niega**.

También se guardan **tres perfiles de ejemplo**, tomados de estudiantes **reales del entrenamiento** (no de la prueba), uno por banda de riesgo, para que quien abra la app empiece desde algo razonable. Se eligen con el propio modelo la fila «de en medio» de cada banda, sin vacíos en las columnas de entrada.

In [4]:
from proyecto.inference.schema import build_schema

completas = train.dropna(subset=COLUMNAS)
p_fail_train = modelo.predict_proba(completas[COLUMNAS])[:, 1]

def perfil(mask) -> dict:
    """Una fila real de la banda, la de en medio (no un extremo), como valores simples."""
    candidatos = completas[mask].sort_values(COLUMNAS[0])
    fila = candidatos.iloc[len(candidatos) // 2]
    return {c: (round(float(fila[c]), 2) if c in NUMERICAS else str(fila[c])) for c in COLUMNAS}

ejemplos = {
    "Perfil de riesgo alto (P(Fail) > 90 %)": perfil(p_fail_train > 0.90),
    "Perfil de riesgo intermedio (P(Fail) entre 40 % y 60 %)": perfil((p_fail_train > 0.40) & (p_fail_train < 0.60)),
    "Perfil de riesgo bajo (P(Fail) < 5 %)": perfil(p_fail_train < 0.05),
}
ETIQUETAS = {
    "previous_exam_score": "Nota del examen previo (0–100)", "previous_gpa": "Promedio previo (1–4)",
    "attendance_percentage": "Asistencia (%)", "assignment_completion_rate": "Tareas entregadas (%)",
    "study_hours_per_day": "Horas de estudio por día", "self_study_hours": "Horas de estudio autónomo",
    "practice_tests_completed": "Pruebas de práctica completadas", "sleep_hours": "Horas de sueño",
    "stress_level": "Nivel de estrés (1–10)", "exam_preparation_days": "Días de preparación del examen",
    "time_management_score": "Manejo del tiempo (0–100)", "exam_anxiety_level": "Ansiedad ante el examen (1–10)",
    "education_level": "Nivel educativo", "school_type": "Tipo de colegio", "parent_education": "Educación de los padres",
    "urban_rural": "Zona", "class_participation": "Participación en clase", "study_consistency": "Constancia de estudio",
    "study_environment": "Ambiente de estudio", "study_method": "Método de estudio", "revision_frequency": "Frecuencia de repaso",
    "notes_quality": "Calidad de los apuntes", "sleep_quality": "Calidad del sueño", "break_frequency": "Frecuencia de descansos",
    "motivation_level": "Motivación", "device_availability": "Dispositivo", "educational_app_usage": "Uso de apps educativas",
    "exam_difficulty": "Dificultad del examen (¿se conoce al avisar?)",
}
USO_RESPONSABLE = (
    "- Es una **alerta para revisión humana**: no decide por sí sola ni debe usarse para sancionar, excluir o limitar oportunidades.\n"
    "- Estima la probabilidad de **no alcanzar 70** a partir de hábitos y datos previos; no es una certeza sobre el futuro del estudiante.\n"
    "- Con costos 5 : 1 (supuesto) avisa a ≈ 90 de cada 100 estudiantes y avisa más a quienes tienen ingreso familiar bajo: revisa cada caso.\n"
    "- Entrenada con un dataset público de Kaggle, posiblemente sintético: no la uses con otra población sin validarla de nuevo.\n"
    "- Límites completos en `models/ficha_modelo.md`."
)
esquema = build_schema(
    train, NUMERICAS, CATEGORICAS, CLASES, cost_matrix=MATRIZ_DE_COSTOS, valid_range=RANGOS_POSIBLES,
    labels=ETIQUETAS, examples=ejemplos, app_title="Alerta de pérdida del curso (pass_70)",
    responsible_use=USO_RESPONSABLE, decision_labels={"Fail": "ALERTA: posible pérdida del curso", "Pass": "Sin alerta"},
    focus_class="Fail", path=MODELS / "input_schema.json",
)
print(f"esquema guardado: {len(NUMERICAS)} numéricas, {len(CATEGORICAS)} categóricas, clases {CLASES}")
print("rangos posibles definidos para:", ", ".join(esquema["valid_range"]))
print("perfiles de ejemplo:", ", ".join(ejemplos))

esquema guardado: 12 numéricas, 16 categóricas, clases ['Pass', 'Fail']
rangos posibles definidos para: previous_exam_score, attendance_percentage, assignment_completion_rate, previous_gpa, sleep_hours, study_hours_per_day, self_study_hours, practice_tests_completed, stress_level, exam_preparation_days, time_management_score, exam_anxiety_level
perfiles de ejemplo: Perfil de riesgo alto (P(Fail) > 90 %), Perfil de riesgo intermedio (P(Fail) entre 40 % y 60 %), Perfil de riesgo bajo (P(Fail) < 5 %)


## 3. Probar la predicción

Tres casos que importan: una fila **normal** responde; una **incompleta** (el usuario marcó «No sé») también responde, porque el Pipeline imputa; una con un dato **imposible** **se niega y explica por qué**. Además se comprueba que la decisión de la app es **la misma regla** de la etapa 7: avisar si P(`Fail`) ≥ 1/6.

In [5]:
from proyecto.inference.predict import InputValidationError, predict_with_scores

normal = pd.DataFrame([ejemplos["Perfil de riesgo intermedio (P(Fail) entre 40 % y 60 %)"]])
print("— fila normal —")
print(predict_with_scores(normal, modelo, esquema).round(3).T.to_string(header=False))

incompleta = normal.copy()
incompleta.loc[:, "previous_exam_score"] = np.nan
print("\n— con un dato faltante (previous_exam_score) —")
print(predict_with_scores(incompleta, modelo, esquema)[["decision", "most_likely", "proba_Fail", "needs_review"]].round(3).to_string(index=False))

for columna, valor in (("study_hours_per_day", 30), ("previous_exam_score", 250), ("previous_gpa", 7)):
    imposible = normal.copy()
    imposible.loc[:, columna] = valor
    print(f"\n— con un dato imposible ({columna} = {valor}) —")
    try:
        predict_with_scores(imposible, modelo, esquema)
        raise AssertionError(f"debía negarse a predecir con {columna} = {valor}")
    except InputValidationError as error:
        print("se niega, como debe:", error)

muestra = completas[COLUMNAS].sample(2000, random_state=0)
salida = predict_with_scores(muestra, modelo, esquema)
misma_regla = (salida["decision"].eq("Fail").to_numpy() == (salida["proba_Fail"].to_numpy() >= UMBRAL)).all()
print("\nla decisión de la app = «avisar si P(Fail) ≥ 1/6» en 2.000 estudiantes de entrenamiento:", bool(misma_regla))
assert misma_regla
print(f"alertas: {salida['decision'].eq('Fail').mean():.1%} · marcados «revisar»: {salida['needs_review'].mean():.1%}")

— fila normal —
decision        Fail
proba_Pass     0.531
proba_Fail     0.469
most_likely     Pass
expected_cost  0.531
needs_review    True

— con un dato faltante (previous_exam_score) —
decision most_likely  proba_Fail  needs_review
    Fail        Fail       0.621         False

— con un dato imposible (study_hours_per_day = 30) —
se niega, como debe: study_hours_per_day: debe estar entre 0 y 24

— con un dato imposible (previous_exam_score = 250) —
se niega, como debe: previous_exam_score: debe estar entre 0 y 100

— con un dato imposible (previous_gpa = 7) —
se niega, como debe: previous_gpa: debe estar entre 1 y 4

la decisión de la app = «avisar si P(Fail) ≥ 1/6» en 2.000 estudiantes de entrenamiento: True
alertas: 90.1% · marcados «revisar»: 20.5%


## 4. Comprobar que el archivo exportado es el modelo que se evaluó

Antes de dárselo a la app: `models/model.joblib` debe dar **las mismas probabilidades** que se obtuvieron en la prueba de la etapa 7. Esto **no es una nueva lectura de la prueba para decidir algo**: es una comprobación de identidad contra lo que ese notebook guardó cuando la abrió (`data/processed/prueba_probabilidades.joblib`).

In [6]:
from proyecto.models.tuning import row_costs

exportado = joblib.load(MODELS / "model.joblib")
guardada = joblib.load(PROCESSED / "prueba_probabilidades.joblib")
test = joblib.load(PROCESSED / "test.joblib")
assert list(test.index) == list(guardada["index"]), "la prueba guardada no es la misma partición"
p_ahora = exportado.predict_proba(test[COLUMNAS])[:, 1]
identico = bool(np.allclose(guardada["p_fail"], p_ahora))
print("probabilidades de la prueba idénticas a las de la etapa 7:", identico, f"(diferencia máxima {np.abs(guardada['p_fail'] - p_ahora).max():.2e})")
assert identico, "El model.joblib exportado no es el modelo que se evaluó"

costo = row_costs(guardada["y_fail"], p_ahora >= UMBRAL).mean()
print(f"costo por estudiante con el exportado: {costo:.4f} · el de la etapa 7: {resultados['prueba']['cost']:.4f}")
assert abs(costo - resultados["prueba"]["cost"]) < 5e-4

probabilidades de la prueba idénticas a las de la etapa 7: True (diferencia máxima 0.00e+00)
costo por estudiante con el exportado: 0.2550 · el de la etapa 7: 0.2550


## 5. La ficha del modelo

La ficha (`models/ficha_modelo.md`) se escribió a mano en la etapa 8 con las cifras de `07_metricas_test.json`. Aquí no se reescribe: se **comprueba** que no tiene campos en blanco y que sus cifras principales coinciden con las del archivo de métricas, para que no pueda contradecir al notebook 07.

In [7]:
ficha = (MODELS / "ficha_modelo.md").read_text(encoding="utf-8")
coma = lambda x, n: f"{x:.{n}f}".replace(".", ",")
p = resultados["prueba"]
cifras = {"costo": coma(p["cost"], 4), "baseline": coma(resultados["baseline_avisar_a_todos"]["cost"], 4),
          "recall de Fail": coma(p["recall_fail"], 3), "NPV": coma(p["npv"], 3)}
print("campos en blanco:", ficha.count("_______________"))
for nombre, texto in cifras.items():
    print(f"  {nombre}: {texto} →", "está en la ficha" if texto in ficha else "FALTA en la ficha")
assert ficha.count("_______________") == 0 and all(t in ficha for t in cifras.values())

campos en blanco: 0
  costo: 0,2550 → está en la ficha
  baseline: 0,3079 → está en la ficha
  recall de Fail: 0,988 → está en la ficha
  NPV: 0,919 → está en la ficha


## 6. Ejecutar la app de Streamlit (`AppTest`)

Se ejecuta **la app real** (`app/streamlit_app.py`) sin abrir un navegador, con la herramienta de pruebas de Streamlit. Para cada perfil de ejemplo: se elige en el selector, se pulsa **Evaluar** y se lee lo que la app muestra. Después se compara con la predicción directa: deben coincidir. También se prueba marcar «No sé» y que el formulario aparece completo.

In [8]:
from datetime import datetime

from streamlit.testing.v1 import AppTest

APP = ROOT / "app" / "streamlit_app.py"

def correr_app(app_path, perfil=None, no_se=None):
    at = AppTest.from_file(str(app_path), default_timeout=90).run()
    assert not at.exception, at.exception
    if perfil:
        at.selectbox(key="example_choice").select(perfil).run()
    if no_se:
        at.checkbox(key=f"unk_num_{no_se}").check().run()
    at.button[0].click().run()
    assert not at.exception, at.exception
    metricas = {m.label: m.value for m in at.metric}
    return {"decision": metricas.get("Resultado sugerido"), "p_fail": metricas.get("Probabilidad de Fail"),
            "revision": bool(at.warning and any("Revisión recomendada" in w.value for w in at.warning)),
            "errores": [e.value for e in at.error]}

vacia = AppTest.from_file(str(APP), default_timeout=90).run()
print("formulario vacío: campos numéricos", len(vacia.number_input), "· listas de categorías", len(vacia.selectbox) - 1, "· casillas «No sé»", len(vacia.checkbox))
assert len(vacia.number_input) == 12 and len(vacia.selectbox) == 17 and len(vacia.checkbox) == 12

pruebas_app = []
for nombre, fila in ejemplos.items():
    directo = predict_with_scores(pd.DataFrame([fila]), modelo, esquema).iloc[0]
    visto = correr_app(APP, perfil=nombre)
    p_app = float(visto["p_fail"].replace("%", "").replace(",", ".")) / 100
    coincide = bool(abs(p_app - directo["proba_Fail"]) < 6e-4)
    pruebas_app.append({"caso": nombre, "p_fail_app": p_app, "decision_app": visto["decision"], "p_fail_directa": float(directo["proba_Fail"]),
                        "decision_directa": directo["decision"], "revision": visto["revision"], "coincide": coincide})
    print(f"{nombre}\n   app: {visto['decision']} · P(Fail) {visto['p_fail']} · revisión: {visto['revision']} | directa: {directo['proba_Fail']:.1%} → coincide: {coincide}")
    assert coincide and not visto["errores"]

con_no_se = correr_app(APP, perfil="Perfil de riesgo intermedio (P(Fail) entre 40 % y 60 %)", no_se="previous_exam_score")
print("\ncon «No sé» en la nota previa →", con_no_se["decision"], "·", con_no_se["p_fail"], "· errores:", con_no_se["errores"])
assert con_no_se["decision"] and not con_no_se["errores"]

2026-11-06 11:48:24.769 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-11-06 11:48:25.196 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


formulario vacío: campos numéricos 12 · listas de categorías 16 · casillas «No sé» 12


2026-11-06 11:48:25.529 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Perfil de riesgo alto (P(Fail) > 90 %)
   app: ALERTA: posible pérdida del curso · P(Fail) 93,8% · revisión: False | directa: 93.8% → coincide: True
Perfil de riesgo intermedio (P(Fail) entre 40 % y 60 %)
   app: ALERTA: posible pérdida del curso · P(Fail) 46,9% · revisión: True | directa: 46.9% → coincide: True


2026-11-06 11:48:25.724 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-11-06 11:48:25.902 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Perfil de riesgo bajo (P(Fail) < 5 %)
   app: Sin alerta · P(Fail) 4,7% · revisión: False | directa: 4.7% → coincide: True

con «No sé» en la nota previa → ALERTA: posible pérdida del curso · 46,9% · errores: []


## 7. Paquete para Databricks Apps

La app se aloja en **Databricks Apps** (tipo Streamlit) sin cambiar su diseño: sigue cargando el modelo desde un archivo y **no consulta MLflow ni el registro** (regla 8 del proyecto: sin credenciales dentro de la app). Para subirla se arma una carpeta limpia con `scripts/prepare_databricks_app.py`: la app, el código de inferencia, `model.joblib`, `input_schema.json`, la ficha, un `app.yaml` y un `requirements.txt` mínimo con las versiones del proyecto. Nunca se copian datos, `.env` ni credenciales.

Para estar seguros de que el paquete **funciona solo**, se ejecuta su app en un proceso nuevo, lejos del repositorio y sin `src/` en el camino: si le faltara algo, fallaría.

In [9]:
import importlib.util
import subprocess

spec = importlib.util.spec_from_file_location("prepare_databricks_app", ROOT / "scripts" / "prepare_databricks_app.py")
preparar = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preparar)

SALIDA = ROOT / "deploy" / "databricks_app"
archivos = preparar.build_bundle(ROOT, SALIDA)
tam = sum((SALIDA / f).stat().st_size for f in archivos) / 1e6
print(f"paquete en deploy/databricks_app: {len(archivos)} archivos, {tam:.2f} MB")
for f in archivos:
    print("  ", f)
assert not any(f.startswith("data") or f.endswith(".env") for f in archivos)
print("\n--- app.yaml ---\n" + (SALIDA / "app.yaml").read_text(encoding="utf-8"))
print("--- requirements.txt ---\n" + (SALIDA / "requirements.txt").read_text(encoding="utf-8"))

paquete en deploy/databricks_app: 10 archivos, 0.04 MB
   app.yaml
   app/streamlit_app.py
   models/ficha_modelo.md
   models/input_schema.json
   models/model.joblib
   requirements.txt
   src/proyecto/__init__.py
   src/proyecto/inference/__init__.py
   src/proyecto/inference/predict.py
   src/proyecto/inference/schema.py

--- app.yaml ---
# Configuración de la app para Databricks Apps.
# Lanza la misma app que se prueba en local: streamlit run app/streamlit_app.py
command: ["streamlit", "run", "app/streamlit_app.py"]
env:
  - name: STREAMLIT_BROWSER_GATHER_USAGE_STATS
    value: "false"

--- requirements.txt ---
pandas==3.0.6
numpy>=2.3,<2.5
scikit-learn==1.9.1
streamlit==1.66.0
joblib==1.6.0



In [10]:
nombre_alto = "Perfil de riesgo alto (P(Fail) > 90 %)"
programa = f"""
import json, sys
from streamlit.testing.v1 import AppTest
at = AppTest.from_file(r"{SALIDA / 'app' / 'streamlit_app.py'}", default_timeout=90).run()
assert not at.exception, at.exception
at.selectbox(key="example_choice").select({nombre_alto!r}).run()
at.button[0].click().run()
assert not at.exception, at.exception
print(json.dumps({{m.label: m.value for m in at.metric}}, ensure_ascii=False))
"""
proceso = subprocess.run([sys.executable, "-I", "-c", programa], capture_output=True, text=True, cwd=str(SALIDA.parent), timeout=300)
print("código de salida:", proceso.returncode)
print(proceso.stdout.strip() or proceso.stderr.strip()[-600:])
assert proceso.returncode == 0, "la app del paquete no corre sola"
desde_paquete = json.loads(proceso.stdout.strip().splitlines()[-1])
esperado = next(x for x in pruebas_app if x["caso"] == nombre_alto)
print("resultado igual al de la app del repositorio:", desde_paquete["Resultado sugerido"] == esperado["decision_app"])
assert desde_paquete["Resultado sugerido"] == esperado["decision_app"]

código de salida: 0
{"Resultado sugerido": "ALERTA: posible pérdida del curso", "Probabilidad de Fail": "93,8%"}
resultado igual al de la app del repositorio: True


## 8. Constancia y siguiente paso

In [11]:
constancia = {
    "fecha_y_hora": datetime.now().isoformat(timespec="seconds"),
    "modelo_exportado_desde": origen,
    "modelo_registrado": f"{registro['modelo_registrado']} v{registro['version']} @{registro['alias']}",
    "model_joblib_kb": round((MODELS / "model.joblib").stat().st_size / 1e3, 1),
    "identico_al_evaluado_en_la_etapa_7": identico,
    "esquema": {"numericas": len(NUMERICAS), "categoricas": len(CATEGORICAS), "con_rango_posible": len(esquema["valid_range"]), "ejemplos": list(ejemplos)},
    "pruebas_app_local": pruebas_app,
    "con_no_se": {"decision": con_no_se["decision"], "p_fail": con_no_se["p_fail"]},
    "paquete_databricks_apps": {"carpeta": "deploy/databricks_app", "archivos": archivos, "corre_solo": True, "mb": round(tam, 3)},
    "despliegue_en_databricks": "pendiente: se hace desde la interfaz de Databricks (carpeta Git + Apps), con la cuenta del usuario; pasos en reports/09_despliegue.md",
}
(REPORTS / "09_inferencia.json").write_text(json.dumps(constancia, indent=2, ensure_ascii=False), encoding="utf-8")
print("guardado reports/09_inferencia.json")
print("modelo exportado:", constancia["modelo_exportado_desde"])

guardado reports/09_inferencia.json
modelo exportado: registro de MLflow (local), estudiantes_aprueba_o_no@champion


## 9. Siguiente: la app

En local (desde la raíz del repositorio):

```bash
streamlit run app/streamlit_app.py
```

En Databricks Apps **sin CLI** (pasos y motivos en `reports/09_despliegue.md`): se sube `deploy/databricks_app/` al repositorio (`git push`); en Databricks se crea una **carpeta Git** (*Git folder*) con el repositorio y se hace *Pull*; en **Compute → Apps** se crea una app personalizada y se despliega apuntando a `.../deploy/databricks_app`.

Comprueba que: el formulario aparece, una predicción normal responde, **«No sé» no rompe nada** y un dato fuera de rango muestra un error claro. Este notebook **no** despliega: eso se hace desde la interfaz de Databricks, con tu cuenta.